q1:
 (a) None
 (b) '2026-05-06'  
 (c) ['2026-05-06', '2026-05-18']   
 (d) [('2026', '05', '06'), ('2026', '05', '18')]   
 (e) ['2026-05-06', '2026-05-18']   
 추가질문:re.findall은 패턴에 캡처 그룹()이 있으면 전체 매치 대신 그룹에 매칭된 부분만 튜플로 반환한다. (c)는 그룹이 없어 전체 문자열을 반환하고 (d)는 3개의 캡처 그룹 때문에 쪼개진 튜플을 반환한다. (e)는 (?:...)를 사용한 비캡처 그룹이므로 그룹이 없는 것처럼 전체 문자열 리스트를 반환한다

q2:
(a) '[T]!'   
(b) '[T]안녕[T] [T]세상[T]!'   
(c) '[T]안녕[T] [T]세상[T]!'   
(d) '수강생 <30>명, 조교 <3>명'   
(e) '수강생 <\x01>명, 조교 <\x01>명' 
추가질문
(i) .+는 가장 길게 매칭하는 탐욕적속성을 가져 첫 <b>부터 마지막 </i>까지 한 번에 치환하지만 .+?는 가장 짧게 매칭하는 Lazy 속성을 가져 각 태그를 개별적으로 치환하기 때문이다
(ii) (d)는 원시 문자열(r"")을 사용해 \1이 1번 그룹으로 잘 해석되지만, (e)는 일반 문자열이어서 \1이 아스키 코드의 8진수 이스케이프 문자로 해석되기 때문이다.

In [ ]:
#q3
import re
from collections import Counter


URL_PAT = re.compile(r"https?://\S+")
TAG_PAT = re.compile(r"<[^>]+>")
EMAIL_PAT = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PAT = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
MSN_TAG_PAT = re.compile(r"[@#]\w+")
KOR_PAT = re.compile(r"[ㄱ-ㅎㅏ-ㅣ]+")
SPACES_PAT = re.compile(r"\s+")
HASH_EXT_PAT = re.compile(r"#([a-zA-Z0-9가-힣]+)")


def clean_post(post: str) -> str:

    post = URL_PAT.sub(" ", post)  
    post = TAG_PAT.sub("", post)  
    post = EMAIL_PAT.sub("[이메일]", post) 
    post = PHONE_PAT.sub("[전화]", post)  
    post = MSN_TAG_PAT.sub(" ", post)  
    post = KOR_PAT.sub("", post)  
    return SPACES_PAT.sub(" ", post).strip()  


def extract_hashtags(post: str) -> list[str]:
    """해시태그 추출 (# 제외)"""  
    return HASH_EXT_PAT.findall(post)


def analyze_posts(posts: list[str]) -> dict:
    """게시물 리스트 분석 후 통계 반환"""  
    n = len(posts)
    cleaned = [clean_post(p) for p in posts]

    
    avg_len = round(sum(len(p) for p in cleaned) / n, 2) if n > 0 else 0.0

    
    tags = []
    for p in posts:
        tags.extend(extract_hashtags(p))
    tag_counts = dict(Counter(tags).most_common()) 

    
    mask_cnt = 0
    for p in posts:
        
        tmp = URL_PAT.sub(" ", p)
        tmp = TAG_PAT.sub("", tmp)

        _, e_cnt = EMAIL_PAT.subn("[이메일]", tmp)  
        _, p_cnt = PHONE_PAT.subn("[전화]", tmp)  
        mask_cnt += e_cnt + p_cnt

    return {
        "posts_n": n, 
        "avg_length_after_clean": avg_len,  
        "hashtag_counts": tag_counts,  
        "masked_count": mask_cnt,  
    }




posts_data: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ",
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠㅠㅠㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15강.",
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    "여러 공백과\n\n\n줄바꿈이 많은 텍스트",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

print("<clean_post 결과>")
for i, p in enumerate(posts_data, 1):
    print(f"Post {i}: {clean_post(p)}")

print("\n<analyze_posts 결과>")
import pprint

pprint.pprint(analyze_posts(posts_data))

<clean_post 결과>
Post 1: 오늘 수업 진짜 재밌었음!! 감사
Post 2: 자료:
Post 3: 팀플 어디서 모이지 카톡
Post 4: 중요: 다음 시험 범위는 1-15강.
Post 5: 문의는 [이메일] ([전화])로!
Post 6: 여러 공백과 줄바꿈이 많은 텍스트
Post 7: 진짜 좋다

<analyze_posts 결과>
{'avg_length_after_clean': 13.57,
 'hashtag_counts': {'DCCP2026': 1, '추천': 1, '팀플': 1, '파이썬': 2},
 'masked_count': 2,
 'posts_n': 7}


<clean_post 결과>
Post 1: 오늘 수업 진짜 재밌었음!! 감사
Post 2: 자료:
Post 3: 팀플 어디서 모이지 카톡
Post 4: 중요: 다음 시험 범위는 1-15강.
Post 5: 문의는 [이메일] ([전화])로!
Post 6: 여러 공백과 줄바꿈이 많은 텍스트
Post 7: 진짜 좋다

<analyze_posts 결과>
{'avg_length_after_clean': 13.57,
 'hashtag_counts': {'DCCP2026': 1, '추천': 1, '팀플': 1, '파이썬': 2},
 'masked_count': 2,
 'posts_n': 7}




설명:정규식 패턴 매칭 속도를 높이기 위해 상단에 re.compile로 패턴을 미리 등록한 후 함수들을 구현하였다. clean_post 함수는 입력된 텍스트를 6단계 순서대로 정제하며 extract_hashtags 함수는 캡처 그룹을 활용해 기호를 뗀 순수 해시태그 이름만 추출한다. 통계를 도출하는 analyze_posts 함수는 정제된 문장들의 평균 글자 수를 반올림하여 계산하고 Counter를 통해 해시태그 빈도를 내림차순 정렬하며 re.subn이 반환하는 치환 횟수를 누적하여 마스킹 건수를 집계한다. 이때 6단계의 순서를 그대로 지켜야 하는 이유는 단계 3과 단계 4의 선후 관계 때문이다. 만약 단계 4를 먼저 수행하면 이메일 주소의 골뱅이 이하 영역이 일반 멘션 패턴에 걸려 공백으로 잘려 나가게 된다. 이로 인해 이메일 구조가 파괴되면 단계 3에서 이메일 패턴을 정상적으로 인식할 수 없어 마스킹 처리가 누락되는 오류가 발생하므로 반드시 주어진 순서를 지켜야한다.